# Tutorial Lab: Advanced Problems with Solutions  
## Sending Exceptions to Python Generators

This notebook is a guided, problem-based tutorial on advanced generator control with:

- `send()`
- `throw()`
- `close()`
- `GeneratorExit`
- `StopIteration`
- `yield from`
- exception translation
- resource cleanup
- generator state machines
- generator-based coroutine protocols

Unlike a compact exercise sheet, each problem is broken into small logical steps:

1. **Mental model**
2. **Problem statement**
3. **Prediction**
4. **Incremental implementation**
5. **Checkpoint tests**
6. **Complete solution**
7. **Explanation**
8. **Common mistakes**
9. **Extension challenge**

All solutions are executable and include assertions.

## How to use this notebook

For each problem:

- Read the explanation before running the code.
- Pause at each **Your prediction** section.
- Try to solve the step before revealing the solution cell.
- Run the checkpoint assertions.
- Read the post-solution explanation carefully.
- Attempt the optional extension before moving on.

The notebook assumes familiarity with ordinary generators and basic exception handling.

## The central mental model

A generator pauses at a `yield` expression.

When the caller invokes:

```python
g.throw(SomeError("message"))
```

Python resumes the generator by raising that exception **at the suspended `yield` expression**.

From there, ordinary exception rules apply:

- a matching `except` block may handle it;
- a `finally` block may run;
- the generator may yield again;
- the generator may return;
- the exception may propagate;
- a different exception may be raised.

## Shared imports and helpers

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from inspect import getgeneratorlocals, getgeneratorstate
from typing import Any, Generator, Iterable

In [2]:
def prime(gen: Generator) -> Generator:
    """Advance a generator to its first yield and return it."""
    next(gen)
    return gen


def gen_state(gen: Generator) -> str:
    """Return the current inspect state of a generator."""
    return getgeneratorstate(gen)


def call_and_capture(func, *args, **kwargs):
    """Return (result, exception) without stopping notebook execution."""
    try:
        return func(*args, **kwargs), None
    except BaseException as exc:
        return None, exc


print("Helpers ready.")

Helpers ready.


# Problem 1 — Locate the exact exception injection point

## Goal

Understand precisely where `throw()` injects an exception.

We will create a generator with multiple statements surrounding a `yield`.  
The event log will reveal which statements execute before and after injection.

## Step 1 — Study the suspension point

Consider this pattern:

```python
log.append("before yield")
value = yield "READY"
log.append("after yield")
```

After `next(g)`, the generator is suspended at the `yield`.

The assignment to `value` has **not** completed yet.

## Your prediction

After priming the generator, suppose the caller executes:

```python
g.throw(ValueError("boom"))
```

Predict:

1. Will `"after yield"` be added to the log?
2. Will `value` receive anything?
3. Will the generator close if the exception is not caught?

## Step 2 — Instrument the generator

In [3]:
def injection_probe(log: list[str]):
    log.append("generator started")
    try:
        while True:
            log.append("before yield")
            value = yield "READY"
            log.append(f"after yield: {value!r}")
    finally:
        log.append("finally cleanup")

## Step 3 — Run the experiment

In [4]:
log = []
g = injection_probe(log)

first = next(g)
assert first == "READY"
assert log == [
    "generator started",
    "before yield",
]

result, exc = call_and_capture(g.throw, ValueError("boom"))

print("result:", result)
print("exception:", repr(exc))
print("log:", log)
print("state:", gen_state(g))

result: None
exception: ValueError('boom')
log: ['generator started', 'before yield', 'finally cleanup']
state: GEN_CLOSED


## Step 4 — Check the result

In [5]:
assert result is None
assert isinstance(exc, ValueError)
assert log == [
    "generator started",
    "before yield",
    "finally cleanup",
]
assert gen_state(g) == "GEN_CLOSED"

print("Checkpoint passed.")

Checkpoint passed.


## Why the solution works

The exception is raised at the suspended `yield` expression.

Therefore:

- the assignment to `value` never completes;
- the statement after the `yield` never runs;
- the `finally` block runs during stack unwinding;
- the uncaught `ValueError` propagates to the caller;
- the generator becomes closed.

## Common mistake

A frequent misconception is that `throw()` raises the exception *after* the `yield`.

It does not.

The exception replaces the normal completion of the `yield` expression.

## Extension challenge

Modify the generator so that it catches `ValueError`, logs the message, and then resumes waiting for another value.

# Problem 2 — Recover from an injected exception without closing

## Goal

Build a generator that treats a specific exception as a recoverable control signal.

The generator should:

- accept ordinary values;
- catch `RecoverableSignal`;
- yield a recovery acknowledgement;
- continue operating afterward.

## Step 1 — Define a narrow control signal

Custom exception classes make a generator protocol easier to understand.

We will avoid catching broad types such as `Exception`.

In [6]:
class RecoverableSignal(Exception):
    """A non-fatal signal understood by the generator."""

## Step 2 — Think about suspension points

The solution will contain two `yield` expressions:

1. the main input `yield`;
2. the acknowledgement `yield` inside the exception handler.

That means the caller must understand which suspension point is active.

## Your prediction

After this call:

```python
ack = g.throw(RecoverableSignal("retry"))
```

the generator will be suspended at the acknowledgement `yield`.

What should the caller do before sending the next ordinary data item?

## Step 3 — Implement the recoverable coroutine

In [7]:
def recoverable_sink(events: list[tuple[str, Any]]):
    while True:
        try:
            item = yield "WAITING"
            events.append(("data", item))
        except RecoverableSignal as exc:
            events.append(("recovery", str(exc)))
            yield f"ACK:{exc}"

## Step 4 — Exercise the protocol carefully

In [8]:
events = []
g = recoverable_sink(events)

assert next(g) == "WAITING"
assert g.send("alpha") == "WAITING"

ack = g.throw(RecoverableSignal("retry-alpha"))
assert ack == "ACK:retry-alpha"
assert gen_state(g) == "GEN_SUSPENDED"

# Leave the acknowledgement yield and return to the main input yield.
assert next(g) == "WAITING"

assert g.send("beta") == "WAITING"

g.close()

print("ack:", ack)
print("events:", events)

ack: ACK:retry-alpha
events: [('data', 'alpha'), ('recovery', 'retry-alpha'), ('data', 'beta')]


## Step 5 — Verify the event sequence

In [9]:
assert events == [
    ("data", "alpha"),
    ("recovery", "retry-alpha"),
    ("data", "beta"),
]

print("Checkpoint passed.")

Checkpoint passed.


## Why the solution works

When `RecoverableSignal` is injected:

1. the main `yield` raises the exception;
2. the matching handler executes;
3. the handler yields an acknowledgement;
4. that acknowledgement becomes the return value of `throw()`;
5. the generator remains suspended;
6. `next(g)` completes the acknowledgement yield and returns execution to the main loop.

## Common mistake

Calling `g.send("beta")` immediately after receiving the acknowledgement would send `"beta"` into the acknowledgement `yield`, not the main input `yield`.

The value would be ignored because the acknowledgement `yield` is not assigned to a variable.

## Better API design

A wrapper object can hide the extra `next(g)` call.  
Later problems will demonstrate this pattern.

# Problem 3 — Return a final value through `StopIteration.value`

## Goal

Use an exception signal to finalize a generator and return a computed result.

The generator will:

- accept numeric values;
- maintain a running total;
- catch `FinalizeTotal`;
- return the total.

## Step 1 — Understand generator return values

Inside a generator:

```python
return result
```

terminates the generator by raising:

```python
StopIteration(result)
```

The caller can inspect the result through:

```python
exc.value
```

In [10]:
class FinalizeTotal(Exception):
    pass

## Step 2 — Build the accumulator

In [11]:
def total_accumulator():
    total = 0

    while True:
        try:
            value = yield total
        except FinalizeTotal:
            return total
        else:
            total += value

## Step 3 — Drive it incrementally

In [12]:
g = total_accumulator()

assert next(g) == 0
assert g.send(5) == 5
assert g.send(8) == 13
assert g.send(-2) == 11

result, exc = call_and_capture(g.throw, FinalizeTotal())

print("ordinary result:", result)
print("captured exception:", repr(exc))

ordinary result: None
captured exception: StopIteration(11)


## Step 4 — Extract the returned value

In [13]:
assert result is None
assert isinstance(exc, StopIteration)
assert exc.value == 11
assert gen_state(g) == "GEN_CLOSED"

print("Final total:", exc.value)

Final total: 11


## Why the solution works

The injected `FinalizeTotal` is caught inside the generator.

The handler executes:

```python
return total
```

A generator cannot return to the caller in the ordinary function-call sense, because it is resumed through `throw()`.

Instead, Python communicates the return value through `StopIteration.value`.

## Common mistake

Catching only `FinalizeTotal` around the caller's `throw()` call will not work.

The generator catches `FinalizeTotal`.  
The caller receives `StopIteration`.

## Extension challenge

Modify the generator so that finalization returns a dictionary containing:

- total;
- count;
- arithmetic mean;
- minimum;
- maximum.

# Problem 4 — Translate a low-level failure into a domain failure

## Goal

Use exception chaining correctly.

A generator receives a low-level exception such as `StorageFailure` and converts it into a higher-level `PipelineFailure`.

## Step 1 — Why translate exceptions?

Low-level exceptions may expose implementation details.

A domain-level exception can provide:

- a stable public API;
- a clearer message;
- preserved diagnostic context.

In [14]:
class StorageFailure(Exception):
    pass


class PipelineFailure(Exception):
    pass

## Step 2 — Choose the chaining strategy

Use:

```python
raise PipelineFailure(...) from exc
```

This sets the original exception as `__cause__`.

In [15]:
def translating_pipeline():
    try:
        while True:
            item = yield "READY"
            print(f"processed {item!r}")
    except StorageFailure as exc:
        raise PipelineFailure("pipeline could not persist the current item") from exc

## Step 3 — Inject the low-level failure

In [16]:
g = translating_pipeline()
assert next(g) == "READY"

_, exc = call_and_capture(
    g.throw,
    StorageFailure("disk unavailable"),
)

print("raised:", repr(exc))
print("cause:", repr(exc.__cause__))

raised: PipelineFailure('pipeline could not persist the current item')
cause: StorageFailure('disk unavailable')


## Step 4 — Verify chaining

In [17]:
assert isinstance(exc, PipelineFailure)
assert isinstance(exc.__cause__, StorageFailure)
assert str(exc.__cause__) == "disk unavailable"
assert gen_state(g) == "GEN_CLOSED"

print("Checkpoint passed.")

Checkpoint passed.


## Why the solution works

The injected exception is caught at the generator's suspended `yield`.

The generator deliberately raises a different exception.

Because `from exc` is used:

- the caller receives `PipelineFailure`;
- the original `StorageFailure` remains available;
- tracebacks explain the causal relationship.

## Common mistake

Using:

```python
raise PipelineFailure("...")
```

inside the handler still creates implicit context, but `from exc` makes the intended cause explicit and easier to inspect.

# Problem 5 — Compare `close()` and `throw(GeneratorExit)`

## Goal

Understand the subtle difference between:

```python
g.close()
```

and:

```python
g.throw(GeneratorExit)
```

## Step 1 — Shared behavior

Both operations inject `GeneratorExit` at the suspension point.

Both normally cause cleanup code to run.

## Step 2 — Important difference

`close()` is a specialized API.

When the generator exits normally after receiving `GeneratorExit`, `close()` suppresses the internal termination details and returns `None`.

A direct `throw(GeneratorExit)` call may expose an exception to the caller.

In [18]:
def cleanup_probe(log: list[str]):
    try:
        while True:
            yield "ACTIVE"
    finally:
        log.append("cleanup")

## Step 3 — Test `close()`

In [19]:
close_log = []
g1 = cleanup_probe(close_log)

assert next(g1) == "ACTIVE"
close_result = g1.close()

assert close_result is None
assert close_log == ["cleanup"]
assert gen_state(g1) == "GEN_CLOSED"

print("close() returned:", close_result)

close() returned: None


## Step 4 — Test direct `throw(GeneratorExit)`

In [20]:
throw_log = []
g2 = cleanup_probe(throw_log)

assert next(g2) == "ACTIVE"
_, exc = call_and_capture(g2.throw, GeneratorExit())

assert isinstance(exc, GeneratorExit)
assert throw_log == ["cleanup"]
assert gen_state(g2) == "GEN_CLOSED"

print("direct throw propagated:", type(exc).__name__)

direct throw propagated: GeneratorExit


## Why the solution works

The generator itself sees `GeneratorExit` in both cases.

The difference is mostly in the caller-facing behavior:

- `close()` is intended for graceful shutdown;
- `throw(GeneratorExit)` behaves like a direct exception injection.

## Important inheritance fact

`GeneratorExit` inherits from `BaseException`, not `Exception`.

Therefore this does not catch it:

```python
except Exception:
    ...
```

## Best practice

Use `close()` for normal generator shutdown.

Use `throw()` for application-specific signals or deliberate testing of exception paths.

# Problem 6 — Detect illegal yielding during shutdown

## Goal

Understand why yielding during `GeneratorExit` is prohibited.

We will intentionally write an incorrect generator.

## Step 1 — The rule

When `close()` injects `GeneratorExit`, the generator must:

- return;
- allow `GeneratorExit` to propagate;
- perform cleanup without yielding.

If it yields a value, Python raises:

```python
RuntimeError: generator ignored GeneratorExit
```

In [21]:
def incorrect_shutdown_generator():
    try:
        yield "OPEN"
    except GeneratorExit:
        # Deliberately incorrect:
        yield "I SHOULD NOT BE YIELDED"

## Step 2 — Trigger the error

In [22]:
g = incorrect_shutdown_generator()
assert next(g) == "OPEN"

_, exc = call_and_capture(g.close)

print(type(exc).__name__ + ":", exc)
print("state after failed close:", gen_state(g))

RuntimeError: generator ignored GeneratorExit
state after failed close: GEN_SUSPENDED


## Step 3 — Verify the behavior

In [23]:
assert isinstance(exc, RuntimeError)
assert "ignored GeneratorExit" in str(exc)
assert gen_state(g) == "GEN_SUSPENDED"

# The generator is suspended at the illegal yield.
# A second close injects GeneratorExit at that new suspension point.
g.close()

assert gen_state(g) == "GEN_CLOSED"
print("Final state:", gen_state(g))

Final state: GEN_CLOSED


## Why the generator remains suspended after the first close

The illegal `yield` actually produced a suspension point before Python reported the runtime error.

The generator therefore remains suspended at that new `yield`.

## Correct pattern

In [24]:
def correct_shutdown_generator(log: list[str]):
    try:
        yield "OPEN"
    finally:
        log.append("release resources")


log = []
g = correct_shutdown_generator(log)
assert next(g) == "OPEN"

g.close()

assert log == ["release resources"]
assert gen_state(g) == "GEN_CLOSED"

print("Correct cleanup:", log)

Correct cleanup: ['release resources']


## Common mistake

Do not use a cleanup handler to produce a final acknowledgement during `close()`.

If a final result is needed, use a separate application-specific signal and return the value through `StopIteration.value`.

# Problem 7 — Build a transactional generator step by step

## Goal

Create a generator-based transaction manager with:

- ordinary data sent through `send()`;
- `CommitTransaction`;
- `RollbackTransaction`;
- cleanup of uncommitted data on close.

## Step 1 — Separate durable and pending state

We need two collections:

- `persisted`: data already committed;
- `pending`: data in the current transaction.

In [25]:
class CommitTransaction(Exception):
    pass


class RollbackTransaction(Exception):
    pass

## Step 2 — Define the snapshot format

Returning a snapshot at every suspension point makes the protocol observable and testable.

In [26]:
@dataclass(frozen=True)
class TransactionSnapshot:
    persisted: tuple[Any, ...]
    pending: tuple[Any, ...]

## Step 3 — Implement only ordinary writes first

The first version does not yet support commit or rollback.

In [27]:
def write_only_transaction(persisted: list[Any]):
    pending: list[Any] = []

    while True:
        value = yield TransactionSnapshot(
            persisted=tuple(persisted),
            pending=tuple(pending),
        )
        pending.append(value)

## Step 4 — Verify the basic data path

In [28]:
storage = []
g = write_only_transaction(storage)

assert next(g) == TransactionSnapshot((), ())
assert g.send("A") == TransactionSnapshot((), ("A",))
assert g.send("B") == TransactionSnapshot((), ("A", "B"))

g.close()

print("Basic write path works.")

Basic write path works.


## Step 5 — Add control exceptions

The complete solution needs an inner `try/except` around the `yield`.

That placement is important: it allows the injected control signal to be caught while keeping the outer loop alive.

In [29]:
def transaction_manager(persisted: list[Any]):
    pending: list[Any] = []

    def snapshot():
        return TransactionSnapshot(
            persisted=tuple(persisted),
            pending=tuple(pending),
        )

    try:
        while True:
            try:
                value = yield snapshot()
                pending.append(value)
            except CommitTransaction:
                persisted.extend(pending)
                pending.clear()
            except RollbackTransaction:
                pending.clear()
    finally:
        pending.clear()

## Step 6 — Walk through a commit

In [30]:
storage = []
tx = transaction_manager(storage)

assert next(tx) == TransactionSnapshot((), ())
assert tx.send("A") == TransactionSnapshot((), ("A",))
assert tx.send("B") == TransactionSnapshot((), ("A", "B"))

after_commit = tx.throw(CommitTransaction())

assert after_commit == TransactionSnapshot(
    persisted=("A", "B"),
    pending=(),
)

print("After commit:", after_commit)

After commit: TransactionSnapshot(persisted=('A', 'B'), pending=())


## Step 7 — Walk through a rollback

In [31]:
assert tx.send("C") == TransactionSnapshot(
    persisted=("A", "B"),
    pending=("C",),
)

after_rollback = tx.throw(RollbackTransaction())

assert after_rollback == TransactionSnapshot(
    persisted=("A", "B"),
    pending=(),
)

print("After rollback:", after_rollback)

After rollback: TransactionSnapshot(persisted=('A', 'B'), pending=())


## Step 8 — Verify cleanup of uncommitted work

In [32]:
assert tx.send("D") == TransactionSnapshot(
    persisted=("A", "B"),
    pending=("D",),
)

tx.close()

assert storage == ["A", "B"]
assert gen_state(tx) == "GEN_CLOSED"

print("Persisted after close:", storage)

Persisted after close: ['A', 'B']


## Why the solution works

The generator uses three distinct paths:

### Data path

```python
value = yield snapshot()
pending.append(value)
```

### Commit path

```python
except CommitTransaction:
    persisted.extend(pending)
    pending.clear()
```

### Rollback path

```python
except RollbackTransaction:
    pending.clear()
```

The `finally` block guarantees that pending work does not survive generator shutdown.

## Common mistake

Placing one large `try/except` outside the loop often causes the generator to exit after handling a single control exception.

For reusable signals, put the exception handler inside the loop.

## Extension challenge

Add a `Savepoint` signal and a `RollbackToSavepoint` signal.

# Problem 8 — Forward exceptions through `yield from`

## Goal

Learn how delegation affects `send()`, `throw()`, `close()`, and returned values.

We will build:

- a child accumulator;
- a parent delegator;
- a reset signal;
- a finish signal.

In [33]:
class ResetAccumulator(Exception):
    pass


class FinishAccumulator(Exception):
    pass

## Step 1 — Implement the child

The child:

- accepts numbers;
- resets on `ResetAccumulator`;
- returns its total on `FinishAccumulator`;
- records cleanup.

In [34]:
def child_accumulator(log: list[Any]):
    total = 0

    try:
        while True:
            try:
                value = yield total
            except ResetAccumulator:
                total = 0
            except FinishAccumulator:
                return total
            else:
                total += value
    finally:
        log.append("child cleanup")

## Step 2 — Implement the parent

The parent delegates with:

```python
result = yield from child_accumulator(log)
```

`yield from` automatically forwards compatible operations to the child.

In [35]:
def parent_delegator(log: list[Any]):
    try:
        child_result = yield from child_accumulator(log)
        log.append(("child returned", child_result))
        return child_result
    finally:
        log.append("parent cleanup")

## Step 3 — Send ordinary values through the parent

In [36]:
log = []
g = parent_delegator(log)

assert next(g) == 0
assert g.send(4) == 4
assert g.send(6) == 10

print("Values were delegated successfully.")

Values were delegated successfully.


## Step 4 — Inject a handled child signal through the parent

In [37]:
reset_result = g.throw(ResetAccumulator())

assert reset_result == 0
assert gen_state(g) == "GEN_SUSPENDED"

print("Reset result:", reset_result)

Reset result: 0


## Step 5 — Finish the child and capture the delegated return value

In [38]:
assert g.send(9) == 9

_, exc = call_and_capture(g.throw, FinishAccumulator())

assert isinstance(exc, StopIteration)
assert exc.value == 9
assert log == [
    "child cleanup",
    ("child returned", 9),
    "parent cleanup",
]
assert gen_state(g) == "GEN_CLOSED"

print("Returned through delegation:", exc.value)
print("Cleanup log:", log)

Returned through delegation: 9
Cleanup log: ['child cleanup', ('child returned', 9), 'parent cleanup']


## Why the solution works

`yield from` acts as a protocol bridge.

It forwards:

- values from the child to the caller;
- values sent by the caller to the child;
- exceptions thrown by the caller to the child;
- close requests to the child.

When the child returns, `yield from` captures `StopIteration.value` and assigns it to `child_result`.

## Common mistake

Manually writing a forwarding loop is difficult to get right because it must correctly handle:

- `send`;
- `throw`;
- `close`;
- return values;
- cleanup.

Use `yield from` for delegation unless there is a specific reason not to.

# Problem 9 — Build a pause/resume state machine

## Goal

Use custom exceptions to control a generator's state without terminating it.

Signals:

- `PauseMachine`
- `ResumeMachine`
- `ResetMachine`
- `StopMachine`

In [39]:
class PauseMachine(Exception):
    pass


class ResumeMachine(Exception):
    pass


class ResetMachine(Exception):
    pass


class StopMachine(Exception):
    pass

## Step 1 — Define the machine snapshot

In [40]:
@dataclass(frozen=True)
class MachineSnapshot:
    total: int
    paused: bool
    accepted: int
    ignored: int

## Step 2 — Decide how paused input behaves

This design will not raise an error for data sent while paused.

Instead:

- the value is ignored;
- `ignored` is incremented.

This keeps the generator alive and makes the behavior explicit.

In [41]:
def controlled_machine():
    total = 0
    paused = False
    accepted = 0
    ignored = 0

    def snapshot():
        return MachineSnapshot(total, paused, accepted, ignored)

    while True:
        try:
            value = yield snapshot()

            if paused:
                ignored += 1
            else:
                total += int(value)
                accepted += 1

        except PauseMachine:
            paused = True

        except ResumeMachine:
            paused = False

        except ResetMachine:
            total = 0
            accepted = 0
            ignored = 0

        except StopMachine:
            return snapshot()

## Step 3 — Exercise every state transition

In [42]:
g = controlled_machine()

assert next(g) == MachineSnapshot(0, False, 0, 0)

assert g.send(5) == MachineSnapshot(5, False, 1, 0)

assert g.throw(PauseMachine()) == MachineSnapshot(5, True, 1, 0)

assert g.send(100) == MachineSnapshot(5, True, 1, 1)
assert g.send(200) == MachineSnapshot(5, True, 1, 2)

assert g.throw(ResumeMachine()) == MachineSnapshot(5, False, 1, 2)

assert g.send(7) == MachineSnapshot(12, False, 2, 2)

print("State transitions passed.")

State transitions passed.


## Step 4 — Reset and stop

In [43]:
assert g.throw(ResetMachine()) == MachineSnapshot(0, False, 0, 0)

_, exc = call_and_capture(g.throw, StopMachine())

assert isinstance(exc, StopIteration)
assert exc.value == MachineSnapshot(0, False, 0, 0)
assert gen_state(g) == "GEN_CLOSED"

print("Final snapshot:", exc.value)

Final snapshot: MachineSnapshot(total=0, paused=False, accepted=0, ignored=0)


## Why this design is robust

The state machine:

- catches only known control signals;
- uses ordinary values for ordinary data;
- keeps every state transition observable;
- returns a final state on explicit shutdown.

## Extension challenge

Add a `SetMultiplier(factor)` control signal.

Because exception instances can carry attributes, the signal can transport configuration data.

# Problem 10 — Carry structured data inside exception signals

## Goal

Use an exception instance as a structured control message.

We will create a `ConfigureWindow` signal containing a new window size.

## Step 1 — Define a data-carrying exception

In [44]:
class ConfigureWindow(Exception):
    def __init__(self, size: int):
        if size <= 0:
            raise ValueError("window size must be positive")
        self.size = size
        super().__init__(f"configure window to {size}")

## Step 2 — Build a moving-window collector

The generator stores recent values.

When `ConfigureWindow(size)` is injected, it updates the maximum retained history.

In [45]:
def window_collector(initial_size: int):
    if initial_size <= 0:
        raise ValueError("initial_size must be positive")

    size = initial_size
    values: list[Any] = []

    def snapshot():
        return {
            "size": size,
            "values": tuple(values),
        }

    while True:
        try:
            value = yield snapshot()
            values.append(value)
            if len(values) > size:
                del values[:-size]
        except ConfigureWindow as signal:
            size = signal.size
            if len(values) > size:
                del values[:-size]

## Step 3 — Send data, then reconfigure

In [46]:
g = window_collector(initial_size=3)

assert next(g) == {"size": 3, "values": ()}
assert g.send("A") == {"size": 3, "values": ("A",)}
assert g.send("B") == {"size": 3, "values": ("A", "B")}
assert g.send("C") == {"size": 3, "values": ("A", "B", "C")}
assert g.send("D") == {"size": 3, "values": ("B", "C", "D")}

smaller = g.throw(ConfigureWindow(2))

assert smaller == {
    "size": 2,
    "values": ("C", "D"),
}

print("After shrinking:", smaller)

After shrinking: {'size': 2, 'values': ('C', 'D')}


## Step 4 — Expand the window and continue

In [47]:
larger = g.throw(ConfigureWindow(5))

assert larger == {
    "size": 5,
    "values": ("C", "D"),
}

assert g.send("E") == {
    "size": 5,
    "values": ("C", "D", "E"),
}

g.close()

print("After expansion:", larger)

After expansion: {'size': 5, 'values': ('C', 'D')}


## Why the solution works

Exceptions are objects.

A custom exception can carry:

- identifiers;
- numeric settings;
- metadata;
- original errors;
- retry information.

However, use this technique only for genuine out-of-band control.  
Ordinary configuration APIs are often clearer when no generator suspension protocol is needed.

# Problem 11 — Inspect live generator state for debugging

## Goal

Use:

- `getgeneratorstate`;
- `getgeneratorlocals`.

We will inspect a suspended transaction without advancing it.

## Step 1 — Create live pending state

In [48]:
storage = []
g = transaction_manager(storage)

next(g)
g.send("item-1")
g.send("item-2")

assert gen_state(g) == "GEN_SUSPENDED"

## Step 2 — Inspect local variables

In [49]:
locals_before = getgeneratorlocals(g)

print("Generator locals:")
for name, value in locals_before.items():
    print(f"  {name} = {value!r}")

Generator locals:
  persisted = []
  snapshot = <function transaction_manager.<locals>.snapshot at 0x00000230A45D1760>
  value = 'item-2'
  pending = ['item-1', 'item-2']


## Step 3 — Verify that inspection is non-destructive

In [50]:
assert locals_before["pending"] == ["item-1", "item-2"]
assert storage == []
assert gen_state(g) == "GEN_SUSPENDED"

rollback_snapshot = g.throw(RollbackTransaction())

assert rollback_snapshot == TransactionSnapshot((), ())
assert getgeneratorlocals(g)["pending"] == []

g.close()

print("Inspection did not advance the generator.")

Inspection did not advance the generator.


## Best practice

Introspection is valuable in:

- debugging;
- teaching;
- diagnostics;
- white-box tests.

Application code should usually rely on an explicit public snapshot instead of generator internals.

# Problem 12 — Wrap a fragile generator protocol in a safe class

## Goal

Raw generator protocols are powerful but easy to misuse.

We will build a wrapper that hides:

- priming;
- direct `send`;
- direct `throw`;
- generator state checks.

## Step 1 — Define the wrapper responsibilities

The class will expose:

- `.write(value)`
- `.commit()`
- `.rollback()`
- `.snapshot`
- `.close()`

In [51]:
class TransactionController:
    def __init__(self, persisted: list[Any]):
        self._generator = transaction_manager(persisted)
        self._snapshot = next(self._generator)
        self._closed = False

    @property
    def snapshot(self) -> TransactionSnapshot:
        return self._snapshot

    def _require_open(self):
        if self._closed:
            raise RuntimeError("controller is closed")

    def write(self, value: Any) -> TransactionSnapshot:
        self._require_open()
        self._snapshot = self._generator.send(value)
        return self._snapshot

    def commit(self) -> TransactionSnapshot:
        self._require_open()
        self._snapshot = self._generator.throw(CommitTransaction())
        return self._snapshot

    def rollback(self) -> TransactionSnapshot:
        self._require_open()
        self._snapshot = self._generator.throw(RollbackTransaction())
        return self._snapshot

    def close(self) -> None:
        if not self._closed:
            self._generator.close()
            self._closed = True

## Step 2 — Use the safer API

In [52]:
storage = []
controller = TransactionController(storage)

assert controller.snapshot == TransactionSnapshot((), ())

controller.write("A")
controller.write("B")

assert controller.snapshot == TransactionSnapshot(
    persisted=(),
    pending=("A", "B"),
)

controller.commit()

assert controller.snapshot == TransactionSnapshot(
    persisted=("A", "B"),
    pending=(),
)

controller.write("C")
controller.rollback()

assert controller.snapshot == TransactionSnapshot(
    persisted=("A", "B"),
    pending=(),
)

controller.close()

print("Persisted:", storage)

Persisted: ['A', 'B']


## Step 3 — Verify closed-state protection

In [53]:
_, exc = call_and_capture(controller.write, "D")

assert isinstance(exc, RuntimeError)
assert str(exc) == "controller is closed"
assert storage == ["A", "B"]

print("Closed-state error:", exc)

Closed-state error: controller is closed


## Why the wrapper is better

The public API now expresses intent directly.

Compare:

```python
controller.commit()
```

with:

```python
g.throw(CommitTransaction())
```

The wrapper:

- reduces protocol mistakes;
- centralizes lifecycle checks;
- hides priming;
- makes future implementation changes easier.

# Problem 13 — Build a restartable worker supervisor

## Goal

Coordinate multiple generator workers and recover from controlled failures.

Each worker will:

- accept numbers;
- maintain total and count;
- yield a report on `RequestReport`;
- return its count on `StopWorker`;
- fail permanently on an unrelated exception.

In [54]:
class RequestReport(Exception):
    pass


class StopWorker(Exception):
    pass

## Step 1 — Implement one worker

In [55]:
def metrics_worker(name: str):
    total = 0.0
    count = 0

    while True:
        try:
            value = yield
        except RequestReport:
            yield {
                "name": name,
                "total": total,
                "count": count,
                "mean": total / count if count else None,
            }
        except StopWorker:
            return count
        else:
            total += float(value)
            count += 1

## Step 2 — Hide the worker's two-yield report protocol

A report is yielded inside an exception handler.

After receiving it, the supervisor must advance the worker back to its ordinary input yield.

In [56]:
def start_worker(name: str):
    return prime(metrics_worker(name))


def request_report(worker: Generator):
    report = worker.throw(RequestReport())
    next(worker)
    return report


def stop_worker(worker: Generator):
    _, exc = call_and_capture(worker.throw, StopWorker())

    if not isinstance(exc, StopIteration):
        raise RuntimeError(f"unexpected worker stop result: {exc!r}")

    return exc.value

## Step 3 — Create and drive multiple workers

In [57]:
workers = {
    "east": start_worker("east"),
    "west": start_worker("west"),
}

workers["east"].send(10)
workers["east"].send(20)

workers["west"].send(5)
workers["west"].send(15)
workers["west"].send(25)

reports = {
    name: request_report(worker)
    for name, worker in workers.items()
}

print("Reports:", reports)

Reports: {'east': {'name': 'east', 'total': 30.0, 'count': 2, 'mean': 15.0}, 'west': {'name': 'west', 'total': 45.0, 'count': 3, 'mean': 15.0}}


## Step 4 — Verify reports

In [58]:
assert reports["east"] == {
    "name": "east",
    "total": 30.0,
    "count": 2,
    "mean": 15.0,
}

assert reports["west"] == {
    "name": "west",
    "total": 45.0,
    "count": 3,
    "mean": 15.0,
}

print("Report checkpoint passed.")

Report checkpoint passed.


## Step 5 — Stop every worker safely

In [59]:
final_counts = {
    name: stop_worker(worker)
    for name, worker in workers.items()
}

assert final_counts == {
    "east": 2,
    "west": 3,
}

assert all(
    gen_state(worker) == "GEN_CLOSED"
    for worker in workers.values()
)

print("Final counts:", final_counts)

Final counts: {'east': 2, 'west': 3}


## Why this problem matters

Real generator protocols often need a coordinator.

The supervisor should centralize:

- priming;
- report extraction;
- extra advancement after handler yields;
- shutdown;
- error handling.

## Extension challenge

Modify the supervisor so that a worker is automatically replaced after an unexpected `ValueError`, while preserving a failure log.

# Problem 14 — Capstone: validated batch-processing coroutine

## Goal

Combine the major ideas into a single tutorial-scale design.

The processor supports:

- ordinary records;
- validation;
- commit;
- rollback;
- pause;
- resume;
- rejection reports;
- graceful shutdown;
- cleanup.

## Step 1 — Define control signals

In [60]:
class CommitBatch(Exception):
    pass


class RollbackBatch(Exception):
    pass


class PauseBatch(Exception):
    pass


class ResumeBatch(Exception):
    pass


class ShutdownBatch(Exception):
    pass

## Step 2 — Define immutable snapshots

Snapshots make every transition easy to test.

In [61]:
@dataclass(frozen=True)
class BatchSnapshot:
    persisted: tuple[int, ...]
    pending: tuple[int, ...]
    rejected: tuple[Any, ...]
    paused: bool
    commits: int
    rollbacks: int

## Step 3 — Decide validation behavior

Rules:

- only non-negative integers are accepted;
- invalid values are added to `rejected`;
- values sent while paused are also rejected;
- rejection does not terminate the generator.

## Step 4 — Implement the processor

In [62]:
def batch_processor(persisted: list[int]):
    pending: list[int] = []
    rejected: list[Any] = []
    paused = False
    commits = 0
    rollbacks = 0

    def snapshot():
        return BatchSnapshot(
            persisted=tuple(persisted),
            pending=tuple(pending),
            rejected=tuple(rejected),
            paused=paused,
            commits=commits,
            rollbacks=rollbacks,
        )

    try:
        while True:
            try:
                item = yield snapshot()

                if paused:
                    rejected.append(item)
                elif isinstance(item, bool) or not isinstance(item, int):
                    rejected.append(item)
                elif item < 0:
                    rejected.append(item)
                else:
                    pending.append(item)

            except CommitBatch:
                persisted.extend(pending)
                pending.clear()
                commits += 1

            except RollbackBatch:
                pending.clear()
                rollbacks += 1

            except PauseBatch:
                paused = True

            except ResumeBatch:
                paused = False

            except ShutdownBatch:
                return snapshot()
    finally:
        pending.clear()

## Step 5 — Prime and submit valid records

In [63]:
storage = []
processor = batch_processor(storage)

current = next(processor)
assert current == BatchSnapshot((), (), (), False, 0, 0)

current = processor.send(10)
current = processor.send(20)

assert current == BatchSnapshot(
    persisted=(),
    pending=(10, 20),
    rejected=(),
    paused=False,
    commits=0,
    rollbacks=0,
)

print("Pending valid records:", current)

Pending valid records: BatchSnapshot(persisted=(), pending=(10, 20), rejected=(), paused=False, commits=0, rollbacks=0)


## Step 6 — Submit invalid records

In [64]:
current = processor.send(-1)
current = processor.send("bad")
current = processor.send(True)

assert current == BatchSnapshot(
    persisted=(),
    pending=(10, 20),
    rejected=(-1, "bad", True),
    paused=False,
    commits=0,
    rollbacks=0,
)

print("After validation failures:", current)

After validation failures: BatchSnapshot(persisted=(), pending=(10, 20), rejected=(-1, 'bad', True), paused=False, commits=0, rollbacks=0)


## Step 7 — Commit the valid batch

In [65]:
current = processor.throw(CommitBatch())

assert current == BatchSnapshot(
    persisted=(10, 20),
    pending=(),
    rejected=(-1, "bad", True),
    paused=False,
    commits=1,
    rollbacks=0,
)

print("After commit:", current)

After commit: BatchSnapshot(persisted=(10, 20), pending=(), rejected=(-1, 'bad', True), paused=False, commits=1, rollbacks=0)


## Step 8 — Pause and test paused input

In [66]:
current = processor.throw(PauseBatch())
assert current.paused is True

current = processor.send(99)

assert current == BatchSnapshot(
    persisted=(10, 20),
    pending=(),
    rejected=(-1, "bad", True, 99),
    paused=True,
    commits=1,
    rollbacks=0,
)

print("Paused input was rejected:", current)

Paused input was rejected: BatchSnapshot(persisted=(10, 20), pending=(), rejected=(-1, 'bad', True, 99), paused=True, commits=1, rollbacks=0)


## Step 9 — Resume, add data, then roll back

In [67]:
current = processor.throw(ResumeBatch())
assert current.paused is False

current = processor.send(30)
current = processor.send(40)

assert current.pending == (30, 40)

current = processor.throw(RollbackBatch())

assert current == BatchSnapshot(
    persisted=(10, 20),
    pending=(),
    rejected=(-1, "bad", True, 99),
    paused=False,
    commits=1,
    rollbacks=1,
)

print("After rollback:", current)

After rollback: BatchSnapshot(persisted=(10, 20), pending=(), rejected=(-1, 'bad', True, 99), paused=False, commits=1, rollbacks=1)


## Step 10 — Gracefully shut down and capture the final result

In [68]:
_, exc = call_and_capture(processor.throw, ShutdownBatch())

assert isinstance(exc, StopIteration)
assert exc.value == BatchSnapshot(
    persisted=(10, 20),
    pending=(),
    rejected=(-1, "bad", True, 99),
    paused=False,
    commits=1,
    rollbacks=1,
)
assert storage == [10, 20]
assert gen_state(processor) == "GEN_CLOSED"

print("Final snapshot:", exc.value)

Final snapshot: BatchSnapshot(persisted=(10, 20), pending=(), rejected=(-1, 'bad', True, 99), paused=False, commits=1, rollbacks=1)


## Capstone explanation

This design has three conceptual channels.

### 1. Data channel

Ordinary records arrive with:

```python
processor.send(record)
```

### 2. Control channel

Out-of-band operations arrive through custom exceptions:

```python
processor.throw(CommitBatch())
processor.throw(RollbackBatch())
processor.throw(PauseBatch())
```

### 3. Lifecycle channel

Shutdown is handled by:

- `ShutdownBatch` for a final report;
- `close()` for unconditional cleanup;
- `finally` for resource safety.

## Capstone design review

The solution follows several strong practices:

- custom, narrow control signals;
- explicit immutable snapshots;
- validation without accidental generator termination;
- `finally` cleanup;
- returned shutdown state through `StopIteration.value`;
- no yielding during `GeneratorExit`;
- clear separation between data and control.

# Problem 15 — Test a complete behavior matrix

## Goal

Create a compact matrix demonstrating the four main outcomes of `throw()`.

We want one generator for each case:

1. catches and yields;
2. does not catch;
3. catches and returns;
4. catches and raises another exception.

## Case A — Catch and yield

In [69]:
class SignalA(Exception):
    pass


def case_a():
    while True:
        try:
            yield "READY"
        except SignalA:
            yield "HANDLED"


g = case_a()
assert next(g) == "READY"

assert g.throw(SignalA()) == "HANDLED"
assert gen_state(g) == "GEN_SUSPENDED"

g.close()

print("Case A: throw() returned a yielded value.")

Case A: throw() returned a yielded value.


## Case B — Do not catch

In [70]:
class SignalB(Exception):
    pass


def case_b():
    while True:
        yield "READY"


g = case_b()
assert next(g) == "READY"

_, exc = call_and_capture(g.throw, SignalB())

assert isinstance(exc, SignalB)
assert gen_state(g) == "GEN_CLOSED"

print("Case B:", type(exc).__name__, "propagated.")

Case B: SignalB propagated.


## Case C — Catch and return

In [71]:
class SignalC(Exception):
    pass


def case_c():
    try:
        while True:
            yield "READY"
    except SignalC:
        return "FINAL"


g = case_c()
assert next(g) == "READY"

_, exc = call_and_capture(g.throw, SignalC())

assert isinstance(exc, StopIteration)
assert exc.value == "FINAL"
assert gen_state(g) == "GEN_CLOSED"

print("Case C returned:", exc.value)

Case C returned: FINAL


## Case D — Catch and raise another exception

In [72]:
class SignalD(Exception):
    pass


class ReplacementError(Exception):
    pass


def case_d():
    try:
        while True:
            yield "READY"
    except SignalD as exc:
        raise ReplacementError("translated") from exc


g = case_d()
assert next(g) == "READY"

_, exc = call_and_capture(g.throw, SignalD("original"))

assert isinstance(exc, ReplacementError)
assert isinstance(exc.__cause__, SignalD)
assert gen_state(g) == "GEN_CLOSED"

print("Case D raised:", repr(exc))
print("Cause:", repr(exc.__cause__))

Case D raised: ReplacementError('translated')
Cause: SignalD('original')


## Summary matrix

| Generator behavior after injection | Caller observes | Generator state |
|---|---|---|
| catches and yields | yielded value returned by `throw()` | suspended |
| does not catch | injected exception | closed |
| catches and returns | `StopIteration`, with optional `.value` | closed |
| catches and raises another exception | replacement exception | usually closed |

# Final best-practices checklist

## Protocol design

- Use ordinary values for ordinary data.
- Use custom exceptions only for clearly out-of-band control.
- Name signals by intent: `Commit`, `Rollback`, `Pause`, `Shutdown`.
- Document every suspension point.
- Hide fragile choreography behind wrapper functions or classes.

## Exception handling

- Catch narrow exception classes.
- Do not accidentally swallow unrelated failures.
- Use explicit exception chaining when translating errors.
- Remember that `GeneratorExit` inherits from `BaseException`.

## Lifecycle safety

- Put cleanup in `finally`.
- Never yield while handling shutdown from `close()`.
- Test both normal close and exceptional termination.
- Verify generator state in tests.

## Delegation

- Prefer `yield from` for generator delegation.
- Understand that `send`, `throw`, and `close` can be forwarded.
- Capture delegated return values through the `yield from` expression.

# Additional unsolved tutorial challenges

These challenges intentionally have no solution cells.

## Challenge 1 — Nested transactions

Add savepoints and partial rollback to the transaction manager.

## Challenge 2 — Retry budget

Create a generator that catches `RetryableFailure`, decrements a retry budget, and raises `RetriesExhausted` when the budget reaches zero.

## Challenge 3 — Half-open circuit breaker

Implement `CLOSED`, `OPEN`, and `HALF_OPEN` states using custom signals.

## Challenge 4 — Delegated parser

Create a parent parser that delegates token processing to a child generator and receives final statistics from the child's return value.

## Challenge 5 — Context-manager wrapper

Wrap the transaction controller in a context manager that commits on success and rolls back on error.

## Challenge 6 — Worker restart policy

Write a supervisor that restarts workers after recoverable failures but permanently removes workers after fatal failures.

## Challenge 7 — Protocol tracing

Create a decorator or wrapper that logs every `next`, `send`, `throw`, and `close` operation applied to a generator.

# Closing perspective

`throw()` is not merely an error mechanism.

Inside a carefully designed generator protocol, it can represent:

- cancellation;
- rollback;
- reset;
- commit;
- reconfiguration;
- pause;
- shutdown;
- fault injection;
- domain-level failure propagation.

The key is disciplined protocol design: explicit signals, narrow handlers, observable state, safe cleanup, and tests for every lifecycle path.